<a href="https://colab.research.google.com/github/zhangling297/Substance-use-codes/blob/main/mediator_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Fitting the Outcome and Mediator Models

# load libraries
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

# load the dataset
# Replace this with your actual data loading step
# For example:
# jobs = pd.read_csv("jobs.csv")

# fit the mediator model (linear regression)
model_m = smf.ols(
    formula="job_seek ~ treat + depress1 + econ_hard + sex + age + occp + marital + nonwhite + educ + income",
    data=jobs
).fit()

# fit the outcome model (probit regression)
model_y = smf.glm(
    formula="work1 ~ treat + job_seek + depress1 + econ_hard + sex + age + occp + marital + nonwhite + educ + income",
    data=jobs,
    family=sm.families.Binomial(link=sm.families.links.Probit())
).fit()

# optional: view summaries
print(model_m.summary())
print(model_y.summary())

# Step 2: Conducting Causal Mediation Analysis

from statsmodels.stats.mediation import Mediation

# model_m and model_y should already be defined from Step 1

med = Mediation(
    outcome_model=model_y.model,    # fitted outcome model's underlying model
    mediator_model=model_m.model,   # fitted mediator model's underlying model
    exposure="treat",
    mediator="job_seek"
)

# fit the mediation analysis
# n_rep is the simulation count, analogous to sims = 1000 in R
m_out = med.fit(n_rep=1000)

# summary of the analysis
print(m_out.summary())


# Step 3: Conducting Sensitivity Analysis from Python via rpy2

# pip install rpy2

import rpy2.robjects as ro
from rpy2.robjects.packages import importr

# load R package
mediation = importr("mediation")

# assume you already created equivalent R models named model.m and model.y in R,
# or fit them directly in R through rpy2

# Example R code execution from Python:
ro.r("""
library(mediation)

# sensitivity analysis
s.out <- medsens(
  model.m,
  model.y,
  sims = 1000,
  T = "treat",
  M = "job_seek",
  INT = FALSE,
  DETAIL = FALSE
)

summary(s.out)
plot(s.out)
""")

In [ ]:
# Alternative codes
# ============================================================
# Causal Mediation Analysis in Python
# Step 1: Fit mediator and outcome models
# Step 2: Conduct mediation analysis
# Step 3: Conduct sensitivity analysis via R's mediation package
# ============================================================

# -------------------------
# 0. INSTALLATION NOTES
# -------------------------
# Run these in terminal or notebook once if needed:
#
# pip install pandas numpy statsmodels matplotlib rpy2
#
# You also need R installed on your computer.
# In R, install the mediation package once:
# install.packages("mediation")


# -------------------------
# 1. IMPORT LIBRARIES
# -------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.mediation import Mediation

# rpy2 imports for Step 3
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

# -------------------------
# 2. LOAD DATA
# -------------------------
# OPTION A: Use your own CSV
# jobs = pd.read_csv("jobs.csv")

# OPTION B: Example using an existing dataframe named `jobs`
# Make sure it contains these variables:
# work1, job_seek, treat, depress1, econ_hard, sex, age,
# occp, marital, nonwhite, educ, income

# Example placeholder:
# jobs = pd.read_csv("your_file_path/jobs.csv")

# -------------------------
# 3. BASIC DATA CHECKS
# -------------------------
required_cols = [
    "work1", "job_seek", "treat", "depress1", "econ_hard", "sex",
    "age", "occp", "marital", "nonwhite", "educ", "income"
]

def validate_data(df, required_columns):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only rows with complete data for the analysis variables
    df2 = df[required_columns].dropna().copy()

    # Optional: ensure binary variables are numeric 0/1 if needed
    # Adjust as needed for your real dataset
    # df2["work1"] = df2["work1"].astype(int)
    # df2["treat"] = df2["treat"].astype(int)

    return df2

jobs_clean = validate_data(jobs, required_cols)

print("Data shape after dropping missing values:", jobs_clean.shape)
print(jobs_clean.head())


# -------------------------
# 4. STEP 1:
#    FIT MEDIATOR MODEL
# -------------------------
# R:
# model.m <- lm(job_seek ~ treat + depress1 + econ_hard + sex + age +
#               occp + marital + nonwhite + educ + income, data = jobs)

mediator_formula = """
job_seek ~ treat + depress1 + econ_hard + sex + age + occp +
           marital + nonwhite + educ + income
"""

model_m = smf.ols(
    formula=mediator_formula,
    data=jobs_clean
).fit()

print("\n" + "="*70)
print("Mediator model summary")
print("="*70)
print(model_m.summary())


# -------------------------
# 5. STEP 1:
#    FIT OUTCOME MODEL
# -------------------------
# R:
# model.y <- glm(work1 ~ treat + job_seek + depress1 + econ_hard + sex +
#                age + occp + marital + nonwhite + educ + income,
#                family = binomial(link="probit"), data = jobs)

outcome_formula = """
work1 ~ treat + job_seek + depress1 + econ_hard + sex + age + occp +
        marital + nonwhite + educ + income
"""

model_y = smf.glm(
    formula=outcome_formula,
    data=jobs_clean,
    family=sm.families.Binomial(link=sm.families.links.Probit())
).fit()

print("\n" + "="*70)
print("Outcome model summary")
print("="*70)
print(model_y.summary())


# -------------------------
# 6. STEP 2:
#    MEDIATION ANALYSIS
# -------------------------
# R:
# m.out <- mediate(model.m, model.y, sims = 1000, T = "treat", M = "job_seek")
# summary(m.out)

# statsmodels Mediation.fit supports simulation-based fitting with n_rep,
# analogous to sims in R. :contentReference[oaicite:1]{index=1}

med = Mediation(
    outcome_model=model_y.model,
    mediator_model=model_m.model,
    exposure="treat",
    mediator="job_seek"
)

# method can be "parametric" or "bootstrap"
m_out = med.fit(method="parametric", n_rep=1000)

print("\n" + "="*70)
print("Mediation analysis summary")
print("="*70)
print(m_out.summary())

# Save mediation summary table if available
try:
    mediation_summary_df = pd.DataFrame(m_out.summary())
    mediation_summary_df.to_csv("mediation_summary.csv", index=True)
    print("\nSaved mediation summary to mediation_summary.csv")
except Exception as e:
    print("\nCould not save mediation summary table automatically:", str(e))


# -------------------------
# 7. STEP 3:
#    SENSITIVITY ANALYSIS USING R medsens() THROUGH PYTHON
# -------------------------
# Why this route:
# Python statsmodels handles mediation estimation, but not the same
# rho-based medsens sensitivity analysis. The R package mediation provides
# medsens() specifically for this purpose. :contentReference[oaicite:2]{index=2}

print("\n" + "="*70)
print("Running sensitivity analysis in R via rpy2")
print("="*70)

# Activate conversion only inside a local converter context,
# which is the recommended rpy2 pattern for pandas conversion. :contentReference[oaicite:3]{index=3}
base = importr("base")
utils = importr("utils")
stats = importr("stats")
mediation_r = importr("mediation")

with localconverter(default_converter + pandas2ri.converter):
    r_jobs = ro.conversion.py2rpy(jobs_clean)

# Put dataframe into R global environment
ro.globalenv["jobs_py"] = r_jobs

# Fit models and run mediate + medsens inside R
# For medsens, the documentation indicates it is applied to mediate output. :contentReference[oaicite:4]{index=4}
r_code = """
# Fit mediator model in R
model.m <- lm(
  job_seek ~ treat + depress1 + econ_hard + sex + age + occp +
    marital + nonwhite + educ + income,
  data = jobs_py
)

# Fit outcome model in R
model.y <- glm(
  work1 ~ treat + job_seek + depress1 + econ_hard + sex + age + occp +
    marital + nonwhite + educ + income,
  family = binomial(link = "probit"),
  data = jobs_py
)

# Mediation analysis
m.out <- mediate(
  model.m,
  model.y,
  sims = 1000,
  treat = "treat",
  mediator = "job_seek"
)

# Sensitivity analysis
s.out <- medsens(
  m.out,
  sims = 1000,
  rho.by = 0.1,
  effect.type = "indirect"
)

# Store printed summaries
m_summary_capture <- capture.output(summary(m.out))
s_summary_capture <- capture.output(summary(s.out))

# Build a table for plotting/export
sens_df <- data.frame(
  rho = s.out$rho,
  indirect.effect = s.out$d0
)
"""

ro.r(r_code)

# Pull R text summaries back into Python
m_summary_text = "\n".join(list(ro.r("m_summary_capture")))
s_summary_text = "\n".join(list(ro.r("s_summary_capture")))

print("\n" + "="*70)
print("R mediation package summary (via rpy2)")
print("="*70)
print(m_summary_text)

print("\n" + "="*70)
print("R medsens summary (via rpy2)")
print("="*70)
print(s_summary_text)

# Pull sensitivity dataframe back into Python
with localconverter(default_converter + pandas2ri.converter):
    sens_df = ro.conversion.rpy2py(ro.globalenv["sens_df"])

print("\nSensitivity results preview:")
print(sens_df.head())

# Save the sensitivity results
sens_df.to_csv("sensitivity_results.csv", index=False)
print("\nSaved sensitivity results to sensitivity_results.csv")


# -------------------------
# 8. PLOT SENSITIVITY ANALYSIS
# -------------------------
# R's plot(s.out) produces a sensitivity plot.
# Here we create a practical Python version from the R output.
plt.figure(figsize=(8, 5))
plt.plot(sens_df["rho"], sens_df["indirect.effect"], marker="o")
plt.axhline(0, linestyle="--")
plt.xlabel("Rho")
plt.ylabel("Estimated mediation effect")
plt.title("Sensitivity Analysis of Mediation Effect")
plt.tight_layout()
plt.savefig("sensitivity_plot.png", dpi=300)
plt.show()

print("\nSaved sensitivity plot to sensitivity_plot.png")


# -------------------------
# 9. OPTIONAL: SAVE MODEL COEFFICIENTS
# -------------------------
model_m.params.to_csv("mediator_model_coefficients.csv", header=["coef"])
model_y.params.to_csv("outcome_model_coefficients.csv", header=["coef"])

print("Saved mediator and outcome model coefficients to CSV files.")


# -------------------------
# 10. OPTIONAL: EXPORT FULL TEXT SUMMARIES
# -------------------------
with open("mediation_r_summary.txt", "w", encoding="utf-8") as f:
    f.write(m_summary_text)

with open("sensitivity_r_summary.txt", "w", encoding="utf-8") as f:
    f.write(s_summary_text)

print("Saved R text summaries to mediation_r_summary.txt and sensitivity_r_summary.txt")

statsmodels provides mediation estimation but not a native medsens() equivalent, so the most practical workflow is:

do Steps 1–2 in Python with statsmodels

do Step 3 by calling R from Python using rpy2 and converting the pandas DataFrame into an R data frame with the recommended local converter pattern.